# Diwali Festival Sales Analysis - Phase 1 & 2: Data Cleaning & Preprocessing

**Dataset Source**: Kaggle - Diwali Sales Data  
**Author**: Senior Data Science & Analytics Team  
**Objective**: Ingest raw retail transactions, audit data quality, handle nulls and duplicates, standardize encodings, and perform feature engineering.


In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Load raw dataset with encoding fallback
raw_path = os.path.join('..', 'data', 'raw', 'Diwali_Sales_Data.csv')
try:
    df_raw = pd.read_csv(raw_path, encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv(raw_path, encoding='unicode_escape')

print('Raw shape:', df_raw.shape)
df_raw.head()


### Initial Data Inspection
Let's inspect data types and null counts across all columns.


In [ ]:
print('--- Dataset Info ---')
df_raw.info()
print('\n--- Missing Values Count ---')
print(df_raw.isnull().sum())


### Data Cleaning Steps
1. Remove empty columns `Status` and `unnamed1` (100% null).
2. Identify and drop duplicate rows.
3. Drop 12 records where target variable `Amount` is null (strict zero synthetic generation policy).
4. Standardize strings: replace non-breaking spaces `\xa0` in `State`.
5. Cast data types appropriately.


In [ ]:
# Step 1: Drop empty columns
cols_to_drop = [c for c in ['Status', 'unnamed1'] if c in df_raw.columns]
df = df_raw.drop(columns=cols_to_drop).copy()

# Step 2: Drop duplicates
dups = df.duplicated().sum()
print(f'Duplicates dropped: {dups}')
df = df.drop_duplicates()

# Step 3: Handle missing target values
null_amounts = df['Amount'].isnull().sum()
print(f'Dropping {null_amounts} null Amount rows')
df = df.dropna(subset=['Amount'])

# Step 4: String standardization
df['State'] = df['State'].astype(str).str.replace('\xa0', ' ').str.strip()
df['Cust_name'] = df['Cust_name'].astype(str).str.strip()
df['Occupation'] = df['Occupation'].astype(str).str.strip()
df['Product_Category'] = df['Product_Category'].astype(str).str.strip()
df['Zone'] = df['Zone'].astype(str).str.strip()

# Step 5: Type casting
df['Amount'] = df['Amount'].round().astype('int64')
df['Age'] = df['Age'].astype('int64')
df['Orders'] = df['Orders'].astype('int64')
df['User_ID'] = df['User_ID'].astype('int64')


### Feature Engineering
We create descriptive labels and metrics for downstream EDA and modeling:
- `Gender_Desc`: Human-readable gender ('Female', 'Male')
- `Marital_Status_Desc`: Binary decoding ('Single', 'Married')
- `Unit_Amount`: Spend per order item (`Amount / Orders`)
- `Spend_Tier`: Purchase bracket (`Budget`, `Mid-Range`, `Premium`)


In [ ]:
df['Gender_Desc'] = df['Gender'].map({'F': 'Female', 'M': 'Male'}).fillna('Other')
df['Marital_Status_Desc'] = df['Marital_Status'].map({0: 'Single', 1: 'Married'}).fillna('Unknown')
df['Unit_Amount'] = (df['Amount'] / df['Orders']).round(2)

spend_bins = [0, 5000, 12000, np.inf]
spend_labels = ['Budget (< ₹5,000)', 'Mid-Range (₹5,000 - ₹12,000)', 'Premium (> ₹12,000)']
df['Spend_Tier'] = pd.cut(df['Amount'], bins=spend_bins, labels=spend_labels)

print('Cleaned Shape:', df.shape)
print('Remaining Nulls:', df.isnull().sum().sum())
df.head()


### Persist Processed Data


In [ ]:
out_path = os.path.join('..', 'data', 'processed', 'diwali_sales_cleaned.csv')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df.to_csv(out_path, index=False, encoding='utf-8')
print(f'Successfully saved cleaned dataset ({len(df)} rows) to {out_path}')
